# Laboratorio: lectura e escritura con PySpark

Empregaremos `penguins.csv` para ler e escribir **CSV, JSON, Parquet, ORC e texto en HDFS**, cambiar de formato e gardar os datos nunha táboa **PostgreSQL mediante JDBC**. O ficheiro ten 344 observacións e sete columnas. Conservaremos todas as filas, incluídas as que conteñen valores nulos.

Executa as celas de código en orde nun notebook de Python 3 de JupyterLab do clúster. A configuración de Spark e YARN xa está preparada. As rutas sen protocolo usan o HDFS configurado; as prácticas con S3/SeaweedFS pertencen ao seguinte tema.

## Preparación: clúster e ficheiro de entrada

Desde **PowerShell**, no cartafol `big-data-cluster`, inicia o clúster base e PostgreSQL:

```powershell
docker compose -f compose.base.yml -f compose.postgres.yml up -d
docker compose -f compose.base.yml -f compose.postgres.yml ps
```

Agarda a que os servizos estean preparados e `postgres-serving` figure como `healthy`. Abre JupyterLab en `http://localhost:8888` (ou no porto configurado no `.env`).

Copia o ficheiro **`data/penguins.csv` destes materiais** ao cartafol **`spark-notebooks/data` do host**, compartido con `/home/hadoop/work/data` no contedor. Antes dos exemplos, abre **File → New → Terminal** en JupyterLab e executa estes comandos HDFS:

```bash
hdfs dfs -mkdir -p /data
hdfs dfs -put /home/hadoop/work/data/penguins.csv /data/penguins.csv
hdfs dfs -ls /data
hdfs dfs -head /data/penguins.csv
```

Se o ficheiro xa está subido, abonda con comprobar o listado e a cabeceira. O usuario debe ter permiso para crear `/data` e subir o ficheiro. Os exemplos len **`/data/penguins.csv`** e gardan os resultados nun directorio novo baixo `/tmp`; non modifican o CSV de entrada.

## 1. Iniciar a sesión Spark

A primeira cela de código crea a `SparkSession`. Emprega a configuración do clúster para executar as aplicacións en YARN; non precisamos instalar PySpark no Windows do host.

In [ ]:
from pyspark.sql import SparkSession

# Crear a sesión ou reutilizar a que xa exista no notebook.
spark = (
    SparkSession.builder
    .appName("lab-lectura-escritura-penguins")
    .getOrCreate()
)

# Consultar o motor e o almacenamento que está usando a sesión.
print("Versión de Spark:", spark.version)
print("Xestor de recursos:", spark.sparkContext.master)
print("Sistema de ficheiros:", spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS"))


## 2. Rutas e esquema dos datos

Cada execución completa emprega un identificador distinto para os destinos HDFS e a táboa PostgreSQL. Se queres repetir o laboratorio, volve executar esta cela e as seguintes. Unha ruta de escritura representa un **directorio con ficheiros `part-...`**, non un único ficheiro.

O esquema explícito emprega cadeas para especie, illa e sexo; decimais para as medidas do peteiro e enteiros para a lonxitude da aleta e a masa corporal.

In [ ]:
from uuid import uuid4
from pyspark.sql import functions as F

# Manter a entrada fixa e illar os resultados de cada execución.
execucion = uuid4().hex
entrada = "/data/penguins.csv"
base = f"/tmp/lab-lectura-escritura-spark/{execucion}"
ruta_csv = f"{base}/csv"
ruta_json = f"{base}/json"
ruta_parquet = f"{base}/parquet"
ruta_orc = f"{base}/orc"
ruta_texto = f"{base}/texto"

# Os nomes e a orde coinciden coa cabeceira do ficheiro orixinal.
esquema = (
    "species STRING, island STRING, bill_length_mm DOUBLE, "
    "bill_depth_mm DOUBLE, flipper_length_mm INT, body_mass_g INT, sex STRING"
)
print("Entrada HDFS:", entrada)
print("Directorio de resultados:", base)


## 3. Ler CSV: texto, inferencia e esquema explícito

`header=True` interpreta a primeira liña como nomes de columnas. Sen inferencia nin esquema, CSV devolve todas as columnas como cadeas de texto. `inferSchema=True` examina os datos para decidir os tipos; un esquema explícito permite fixalos directamente.

In [ ]:
# Ler sen inferencia: as medidas tamén aparecen como string.
df_cadeas = spark.read.option("header", True).csv(entrada)
df_cadeas.printSchema()

# Ler inferindo os tipos das columnas numéricas.
df_inferido = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(entrada)
)
df_inferido.printSchema()


In [ ]:
# Empregar o esquema coñecido para todos os exemplos seguintes.
df = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .schema(esquema)
    .csv(entrada)
)
df.show(10, truncate=False)
df.printSchema()
n_entrada = df.count()
print("Observacións:", n_entrada)
assert n_entrada == 344

# Mostrar os nulos sen eliminar ningunha observación.
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()


## 4. Escribir e volver ler CSV

Gardamos unha copia con cabeceira e separador coma. Ao recuperala indicamos de novo o esquema porque CSV conserva os valores como texto. `errorifexists` evita substituír un destino existente.

In [ ]:
# Escribir todas as observacións nun directorio CSV novo.
(
    df.write
    .mode("errorifexists")
    .option("header", True)
    .option("sep", ",")
    .csv(ruta_csv)
)

# Recuperar as columnas cos mesmos tipos do DataFrame orixinal.
df_csv = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .schema(esquema)
    .csv(ruta_csv)
)
df_csv.show(5, truncate=False)
df_csv.printSchema()
print("Filas recuperadas de CSV:", df_csv.count())


## 5. Escribir e ler JSON

Spark escribe un obxecto JSON por liña. Mantemos os campos nulos mediante `ignoreNullFields=False` e indicamos o esquema na lectura. A opción `multiLine=True` úsase para documentos JSON que ocupan varias liñas; non é necesaria para esta saída de Spark.

In [ ]:
# Escribir JSON por liñas, incluíndo os campos con valor null.
df.write.mode("errorifexists").option("ignoreNullFields", "false").json(ruta_json)

# Ler os obxectos JSON cos tipos coñecidos.
df_json = spark.read.schema(esquema).json(ruta_json)
df_json.show(5, truncate=False)
df_json.printSchema()
print("Filas recuperadas de JSON:", df_json.count())


## 6. Escribir e ler Parquet

Parquet é un formato binario columnar que incorpora o esquema. Non precisamos cabeceira, separador nin inferencia ao lelo. A orde das filas recuperadas non está garantida.

In [ ]:
# Gardar os datos e o seu esquema en Parquet.
df.write.mode("errorifexists").parquet(ruta_parquet)

# Recuperar automaticamente os nomes e tipos das columnas.
df_parquet = spark.read.parquet(ruta_parquet)
df_parquet.show(5, truncate=False)
df_parquet.printSchema()
print("Filas recuperadas de Parquet:", df_parquet.count())


## 7. Escribir e ler ORC

ORC tamén é un formato binario columnar e conserva o esquema. Cambia o método de lectura e escritura, mantendo o mesmo DataFrame e almacenamento HDFS.

In [ ]:
# Escribir nun destino ORC independente do destino Parquet.
df.write.mode("errorifexists").orc(ruta_orc)

# Ler ORC sen especificar o esquema desde o notebook.
df_orc = spark.read.orc(ruta_orc)
df_orc.show(5, truncate=False)
df_orc.printSchema()
print("Filas recuperadas de ORC:", df_orc.count())


## 8. Ler e escribir texto

`spark.read.text` devolve unha columna `value` de tipo cadea, cunha fila por liña. Se lemos o CSV como texto, a cabeceira tamén conta como unha liña: esperamos **345 liñas**.

Para escribir texto, o DataFrame debe conter unha única columna de cadeas. Converteremos cada observación nunha cadea JSON e gardarémola co método `.text()`. A lectura de texto recupera cadeas; `from_json` permite interpretalas co esquema coñecido.

In [ ]:
# Ler as liñas orixinais sen interpretar columnas CSV.
liñas_csv = spark.read.text(entrada)
liñas_csv.show(5, truncate=False)
assert liñas_csv.count() == n_entrada + 1

# Representar cada observación nunha soa columna string.
df_texto = df.select(
    F.to_json(
        F.struct(*[F.col(c) for c in df.columns]),
        options={"ignoreNullFields": "false"},
    ).alias("value")
)
df_texto.write.mode("errorifexists").text(ruta_texto)

# Recuperar as cadeas e reconstruír as columnas do DataFrame.
texto_lido = spark.read.text(ruta_texto)
texto_lido.show(3, truncate=False)
df_desde_texto = texto_lido.select(
    F.from_json("value", esquema).alias("penguin")
).select("penguin.*")
df_desde_texto.show(5, truncate=False)


## 9. Cambiar de CSV a Parquet

Lemos a copia CSV e escribimos os mesmos datos noutro directorio Parquet. Cambia a representación, mantendo as columnas e os valores.

In [ ]:
# Converter o CSV recuperado a un novo destino columnar.
ruta_conversion = f"{base}/csv_a_parquet"
df_csv.write.mode("errorifexists").parquet(ruta_conversion)
df_conversion = spark.read.parquet(ruta_conversion)
df_conversion.show(5, truncate=False)
df_conversion.printSchema()


## 10. Comprobar as lecturas e os ficheiros HDFS

Para comprobar que os formatos conservan os datos, comparamos os recontos e as filas en ambos sentidos con `exceptAll`. Así tamén se comproban as repeticións e os nulos, sen depender da orde das filas nin da propiedade `nullable` do esquema.

In [ ]:
# Comprobar as columnas na mesma orde para comparar os valores.
recuperados = {
    "CSV": df_csv,
    "JSON": df_json,
    "Parquet": df_parquet,
    "ORC": df_orc,
    "Texto interpretado": df_desde_texto,
    "CSV a Parquet": df_conversion,
}
for formato, recuperado in recuperados.items():
    recuperado = recuperado.select(*df.columns)
    assert recuperado.count() == n_entrada, formato
    assert df.exceptAll(recuperado).count() == 0, formato
    assert recuperado.exceptAll(df).count() == 0, formato
    print(f"{formato}: {n_entrada} filas, datos conservados")

# Copiar estes comandos ao terminal de JupyterLab para ver os ficheiros.
print(f"hdfs dfs -ls {base}")
print(f"hdfs dfs -ls {ruta_csv}")
print(f"hdfs dfs -ls {ruta_parquet}")


### 10.1. Comparar os esquemas recuperados

Ata agora comprobamos os valores. Agora observaremos os **tipos das columnas**. A seguinte cela compara as tres lecturas do CSV orixinal e as saídas gardadas en distintos formatos. Tamén le o JSON sen esquema explícito para observar os tipos que Spark infire.

En CSV, o esquema explícito e a inferencia son dúas maneiras de recuperar os tipos: o ficheiro non os incorpora. JSON distingue valores de texto, números e nulos, pero a inferencia pode escoller tipos distintos aos do DataFrame orixinal. Parquet e ORC almacenan os tipos das columnas.


In [ ]:
# Ler JSON sen esquema para comparar a inferencia cos tipos coñecidos.
df_json_inferido = spark.read.json(ruta_json)

# Consultar os esquemas existentes sen transformar os datos.
lecturas = {
    "CSV sen inferencia": df_cadeas,
    "CSV con inferencia": df_inferido,
    "CSV con esquema": df,
    "CSV recuperado con esquema": df_csv,
    "JSON con inferencia": df_json_inferido,
    "JSON con esquema": df_json,
    "Parquet": df_parquet,
    "ORC": df_orc,
}

# Mostrar os tipos de cada columna nunha táboa de texto.
cabeceira = f"{'Lectura':30}" + " | ".join(f"{c:20}" for c in df.columns)
print(cabeceira)
print("-" * len(cabeceira))
for nome, datos_lidos in lecturas.items():
    tipos = dict(datos_lidos.dtypes)
    print(f"{nome:30}" + " | ".join(f"{tipos[c]:20}" for c in df.columns))


### 10.2. Comparar os tamaños en HDFS

Executaremos `hdfs dfs -du -s -h` desde o notebook usando o cliente HDFS do contedor. `-s` suma o tamaño de cada destino e `-h` mostra unidades lexibles. A primeira columna indica o **tamaño lóxico dos ficheiros**; a segunda, o **espazo contabilizado coa replicación**. Para comparar os formatos, fíxate na primeira columna.

Mediremos o CSV orixinal e os directorios de saída CSV, JSON, Parquet e ORC. As saídas conteñen as mesmas 344 observacións. O CSV de saída pode ocupar un tamaño distinto ao orixinal pola representación dos valores, o número de ficheiros e as cabeceiras.

É unha comparación ilustrativa cun conxunto pequeno: os metadatos, as particións e a compresión tamén inflúen no tamaño. Non permite concluír que un formato será sempre o máis pequeno nin medir a velocidade de lectura.


In [ ]:
import subprocess

# Medir os ficheiros xa creados; estes comandos só consultan HDFS.
destinos = {
    "CSV orixinal": entrada,
    "CSV escrito": ruta_csv,
    "JSON": ruta_json,
    "Parquet": ruta_parquet,
    "ORC": ruta_orc,
}
print("Formato | tamaño lóxico | espazo coa replicación | ruta")
for formato, ruta in destinos.items():
    comando = ["hdfs", "dfs", "-du", "-s", "-h", ruta]
    resultado = subprocess.run(comando, check=True, capture_output=True, text=True)
    print(f"{formato}: {resultado.stdout.strip()}")


### 10.3. Preguntas para interpretar os resultados

Responde a partir das saídas anteriores:

1. Que tipo ten `body_mass_g` ao ler CSV sen inferencia? Como cambia ao inferir os tipos ou indicar o esquema?
2. Que diferenzas aparecen entre os tipos inferidos de JSON e o esquema explícito, especialmente nas columnas enteiras?
3. Por que podemos recuperar os tipos de Parquet e ORC sen indicar o esquema nin activar a inferencia?
4. Que saída ocupa menos e cal ocupa máis segundo o tamaño lóxico mostrado por HDFS? Que factores poden explicar as diferenzas?
5. Por que o CSV orixinal e a saída CSV poden ocupar tamaños distintos tendo as mesmas observacións?
6. Que ficheiros `part-...` e marcadores auxiliares aparecen nos listados? Por que a ruta de saída é un directorio?
7. As comprobacións con `exceptAll` conservaron tamén os nulos? Por que comparar só o número de filas non abondaría para comprobar os datos?

Engade as respostas nunha cela Markdown baixo estas preguntas. Xustifica as conclusións cos resultados da túa execución.


Os comandos impresos mostran os directorios e os ficheiros `part-...`, así como os marcadores auxiliares que poidan aparecer, por exemplo `_SUCCESS`. O número de ficheiros depende das particións da escritura.

## 11. Modos de escritura

| Modo | Se o destino xa existe |
|---|---|
| `errorifexists` | Produce un erro; é o predeterminado. |
| `ignore` | Conserva o destino e omite a escritura. |
| `append` | Engade os rexistros aos existentes. |
| `overwrite` | Substitúe os datos do destino. |

A seguinte demostración usa un directorio propio. Ao engadir as mesmas observacións con `append` teremos 688 filas; con `overwrite` volveremos a 344. Reexecutar a cela illada falla na primeira escritura porque o directorio xa existe: xera outra `execucion` para repetir todo o laboratorio.

In [ ]:
# Crear un destino exclusivo para demostrar os modos de escritura.
ruta_modos = f"{base}/modos_parquet"
df.write.mode("errorifexists").parquet(ruta_modos)

# Ignore deixa intactos os datos xa gardados.
df.write.mode("ignore").parquet(ruta_modos)
assert spark.read.parquet(ruta_modos).count() == n_entrada

# Append engade as mesmas filas unha segunda vez.
df.write.mode("append").parquet(ruta_modos)
print("Filas tras append:", spark.read.parquet(ruta_modos).count())
assert spark.read.parquet(ruta_modos).count() == 2 * n_entrada

# Overwrite substitúe só este destino de demostración.
df.write.mode("overwrite").parquet(ruta_modos)
print("Filas tras overwrite:", spark.read.parquet(ruta_modos).count())
assert spark.read.parquet(ruta_modos).count() == n_entrada


## 12. Escribir en PostgreSQL mediante JDBC

O servizo do clúster é `postgres-serving`, coa base de datos `gold_serving` e o usuario/contrasinal `gold`/`gold`. Desde Spark usamos **`postgres-serving:5432`**, accesible na rede `cluster`. Un cliente no Windows do host usaría `localhost:5433` por defecto; ese porto pode cambiar mediante `POSTGRES_SERVING_HOST_PORT`.

A imaxe do clúster debe incluír o driver JDBC `org.postgresql.Driver` para o driver de Spark e os executores. O Dockerfile do curso inclúe o JAR PostgreSQL en `/opt/spark/jars-extra` e a configuración engádeo aos classpaths. Non instalamos dependencias desde o notebook.

Spark creará unha táboa nova no esquema `public` e gardará nela as 344 observacións. Non define automaticamente unha clave primaria. A escritura limita as conexións simultáneas a unha, suficiente para este conxunto pequeno.

In [ ]:
# Configurar a conexión ao PostgreSQL interno do clúster.
jdbc_url = "jdbc:postgresql://postgres-serving:5432/gold_serving"
propiedades_jdbc = {
    "user": "gold",
    "password": "gold",
    "driver": "org.postgresql.Driver",
}
taboa_pg = f"public.spark_penguins_{execucion}"

# Crear unha táboa propia e inserir o DataFrame completo.
(
    df.write
    .mode("errorifexists")
    .option("numPartitions", "1")
    .option("batchsize", "1000")
    .jdbc(url=jdbc_url, table=taboa_pg, properties=propiedades_jdbc)
)
print("Táboa PostgreSQL:", taboa_pg)


### Ler a táboa PostgreSQL e comprobar o resultado

Recuperamos a táboa nun DataFrame e comprobamos que conserva as observacións orixinais. `append` en JDBC engade filas e pode duplicalas; non realiza un *upsert*. `overwrite`, por defecto, elimina e recrea a táboa, podendo perder índices e restricións. `ignore` omite a escritura se a táboa xa existe.

In [ ]:
# Ler a táboa gardada mediante o mesmo conector JDBC.
df_pg = spark.read.jdbc(
    url=jdbc_url,
    table=taboa_pg,
    properties=propiedades_jdbc,
).select(*df.columns)
df_pg.show(10, truncate=False)
df_pg.printSchema()

# Comparar todas as filas, incluídas as que teñen campos nulos.
assert df_pg.count() == n_entrada
assert df.exceptAll(df_pg).count() == 0
assert df_pg.exceptAll(df).count() == 0
print("PostgreSQL: datos conservados")

# Copiar estas consultas a psql para comprobar a táboa directamente.
print(f"SELECT count(*) FROM {taboa_pg};")
print(f"SELECT * FROM {taboa_pg} ORDER BY species, island LIMIT 10;")


Desde **PowerShell**, abre o cliente PostgreSQL:

```powershell
docker exec -it postgres-serving psql -U gold -d gold_serving
```

Executa en `psql` as consultas impresas pola cela anterior. O reconto esperado é 344. Para saír, escribe `\q`.

Se aparece `ClassNotFoundException: org.postgresql.Driver`, comproba a imaxe e os classpaths do clúster. Se falla a conexión, comproba o estado de PostgreSQL e a rede compartida co notebook e cos executores.

## 13. Pechar a sesión

Executa esta cela cando remates. Libera os recursos da aplicación Spark; os ficheiros HDFS e a táboa PostgreSQL permanecen gardados.

In [ ]:
# Pechar a sesión despois de todas as lecturas e escrituras.
spark.stop()
